# Agentic Finance Evidence Gate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crowdvector/polybridge-cookbooks/blob/main/agentic-finance/agentic-finance.ipynb)

This notebook runs the Agentic Finance cookbook in offline mode. It uses sanitized fixtures, normalizes PolyBridge-shaped evidence into `EvidencePacket` objects, applies deterministic gate rules, writes memos, and appends redacted JSONL audit records.

It is research/demo software, not financial advice. It does not require API keys, does not call live PolyBridge by default, does not call Alpaca, does not submit orders, and does not create a real-money trading path.

Notebook flow:

1. Set up local or Colab files.
2. Run Tier 1: Evidence Gate offline.
3. Preview `evidence-packet.json` and `decision-memo.md`.
4. Run Tier 2: Portfolio Event-Risk Map offline.
5. Preview `portfolio-risk-map.json` and `portfolio-risk-memo.md`.


## 1. Setup

When running from a local checkout, this cell uses the files already in `agentic-finance/`. In Colab, it downloads the cookbook files from the repository after they are available on `main`.

The optional `requests` dependency supports live PolyBridge mode, but this notebook does not run live mode by default.

In [ ]:
from pathlib import Path
import subprocess
import sys
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/crowdvector/polybridge-cookbooks/main/agentic-finance"


def find_cookbook_dir() -> Path:
    for candidate in (Path("."), Path("agentic-finance")):
        if (candidate / "evidence_gate.py").exists() and (candidate / "agentic_finance").exists():
            return candidate
    return Path(".")


COOKBOOK_DIR = find_cookbook_dir()

FILES = [
    "requirements.txt",
    "evidence_gate.py",
    "run_portfolio_risk_map.py",
    "agentic_finance/__init__.py",
    "agentic_finance/audit.py",
    "agentic_finance/cli.py",
    "agentic_finance/evidence.py",
    "agentic_finance/gate.py",
    "agentic_finance/memo.py",
    "agentic_finance/models.py",
    "agentic_finance/polybridge.py",
    "agentic_finance/portfolio.py",
    "agentic_finance/redaction.py",
    "agentic_finance/brokers/__init__.py",
    "agentic_finance/brokers/alpaca.py",
    "fixtures/thesis.json",
    "fixtures/polybridge_forecast.response.json",
    "fixtures/polybridge_search.response.json",
    "fixtures/portfolio/ai_regulation_forecast.response.json",
    "fixtures/portfolio/ai_regulation_search.response.json",
    "fixtures/portfolio/energy_forecast.response.json",
    "fixtures/portfolio/energy_search.response.json",
    "fixtures/portfolio/rates_forecast.response.json",
    "fixtures/portfolio/rates_search.response.json",
    "fixtures/portfolio/volatility_forecast.response.json",
    "fixtures/portfolio/volatility_search.response.json",
    "examples/sample_holdings.csv",
]

for relative in FILES:
    target = COOKBOOK_DIR / relative
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{BASE_URL}/{relative}", target)

requirements_path = COOKBOOK_DIR / "requirements.txt"
if requirements_path.exists():
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
    except Exception as exc:
        print("Dependency install skipped; offline cells do not require live PolyBridge dependencies.")
        print(type(exc).__name__, exc)

print("Cookbook directory:", COOKBOOK_DIR.as_posix())


## 2. Display Helpers

These helpers load local modules, print output paths, and show short previews without persisting raw provider responses.

In [ ]:
import importlib.util
import json
from pathlib import Path

from IPython.display import Markdown, display


def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def relative_path(path) -> str:
    path = Path(path)
    try:
        return path.resolve().relative_to(COOKBOOK_DIR.resolve()).as_posix()
    except (OSError, RuntimeError, ValueError):
        return path.as_posix()


def display_paths(title: str, paths: dict) -> None:
    lines = [f"- `{label}`: `{relative_path(path)}`" for label, path in paths.items()]
    display(Markdown(f"### {title}\n" + "\n".join(lines)))


def preview_json(title: str, path, max_chars: int = 1800) -> None:
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    text = json.dumps(data, indent=2, sort_keys=True)
    if len(text) > max_chars:
        text = text[:max_chars].rstrip() + "\n..."
    display(Markdown(f"### {title}\n`{relative_path(path)}`\n\n```json\n{text}\n```"))


def preview_markdown(title: str, path, max_chars: int = 1800) -> None:
    text = Path(path).read_text(encoding="utf-8")
    if len(text) > max_chars:
        text = text[:max_chars].rstrip() + "\n..."
    display(Markdown(f"### {title}\n`{relative_path(path)}`\n\n```markdown\n{text}\n```"))


## 3. Tier 1: Evidence Gate Offline

This run uses the committed sanitized fixtures. It creates local review artifacts only. If the deterministic gate clears, the cookbook may create an Alpaca-style paper-preview JSON object, but it does not import the Alpaca SDK, connect to Alpaca, or submit anything.

In [ ]:
evidence_gate = load_module("evidence_gate", COOKBOOK_DIR / "evidence_gate.py")

tier1_result = evidence_gate.run_offline(output_dir=COOKBOOK_DIR / "outputs")

display(Markdown("## Tier 1 Evidence Gate offline run"))
print("Decision:", tier1_result["gate_decision"].decision)
print("Paper preview allowed:", tier1_result["gate_decision"].cleared_for_paper_preview)
display_paths("Generated output paths", tier1_result["paths"])


In [ ]:
preview_json("EvidencePacket preview", tier1_result["paths"]["evidence_packet"])
preview_markdown("DecisionMemo preview", tier1_result["paths"]["decision_memo"])


## 4. Tier 2: Portfolio Event-Risk Map Offline

This run reads the local sample holdings CSV, maps holdings to deterministic event-risk exposures, normalizes fixture-backed evidence into `EvidencePacket` objects, applies the gate per exposure, and writes a portfolio risk map and memo.

The portfolio memo is a risk review artifact only. It does not provide recommendations, order instructions, broker connectivity, or a real-money path.

In [ ]:
portfolio_runner = load_module("run_portfolio_risk_map", COOKBOOK_DIR / "run_portfolio_risk_map.py")

portfolio_result = portfolio_runner.run_portfolio_risk_map_workflow(
    holdings_path=COOKBOOK_DIR / "examples" / "sample_holdings.csv",
    base_dir=COOKBOOK_DIR,
    output_dir=COOKBOOK_DIR / "outputs",
)

display(Markdown("## Tier 2 Portfolio Event-Risk Map offline run"))
print("Exposures:", ", ".join(exposure.exposure_id for exposure in portfolio_result["exposures"]))
display_paths("Generated output paths", portfolio_result["paths"])


In [ ]:
preview_json("Portfolio risk map preview", portfolio_result["paths"]["portfolio_risk_map"])
preview_markdown("Portfolio risk memo preview", portfolio_result["paths"]["portfolio_risk_memo"])


## Notes

Runtime files are written under `outputs/`, which is ignored by git. Before committing notebook changes, code cell outputs and execution counts should remain cleared.